This notebook removes low quality samples and calculates covariate-adjusted Z scores for the expression data.

This notebook should take about 5 minutes to run.

## Setup

In [1]:
library(data.table)
library(parallel)
options(mc.cores=4L) # Adjust based on your VM's specs. Example specs are 4 cores 26 RAM.

## Load data

In [2]:
counts <- fread('mage_expression-counts.csv')
tpm    <- fread('mage_expression-tpm.csv')
covars <- fread('mage_eQTL_covariates.tab.gz')

# Sanity checks
all(names(tpm)==names(counts))
all(tpm$gene  ==counts$gene  )

[1] TRUE

[1] TRUE

## Filter TPM
* Only keep genes where at least 20% of samples have > the minimum readcount (6) or TPM threshold (0.1).
* Only keep genes where at least 5% of samples (rounded up) have TPM > 0.
* Log2 transform.
* Convert to Z-scores (`scale()`).

In [3]:
min_readcount <- 6
min_tpm       <- 0.1
N <- nrow(tpm)

cols2keep <- ( colSums(tpm>min_tpm & counts>min_readcount) > N*0.20 ) &
             ( colSums(tpm>0)                              > N*0.05 )

tpm_norm <- tpm[
  ][, .SD, .SDcols=cols2keep
  ][, names(.SD) := lapply(.SD, function(x) scale(log2(x+2))), .SDcols=is.numeric
]

## Calculate scaled residuals
(A.K.A. covariate-adjusted Z-scores.)\
This code block may take a few minutes to run.

In [4]:
gene_nms  <- grep('^ENSG',names(tpm_norm), value=T)
covar_nms <- names(covars)[-1]
fmlas <- paste(gene_nms,'~',paste(covar_nms,collapse='+')) |> setNames(gene_nms)
data <- covars[tpm_norm, on='sample_id']

tpm_resid <- as.data.table(mclapply(fmlas, function(fmla) lm(fmla,data) |> residuals() |> scale()))

## Write

In [5]:
tpm_resid <- cbind(sample_id=tpm_norm$sample_id, tpm_resid)
fwrite(tpm_resid,'mage_expression-tpm_residuals.csv')
system('gcloud storage cp mage_expression-tpm_residuals.csv $WORKSPACE_BUCKET/data/derived/')